In [1]:
from collections import Counter
from Bio import SeqIO
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

def run_knn_with_features(repeats_folder, train_folder, test_folder, report_file, n_neighbors=5):
    # -----------------------------
    # 1️⃣ Load known repeats
    # -----------------------------
    labels, repeats = [], []
    for file in os.listdir(repeats_folder):
        if file.startswith("."):
            continue
        with open(os.path.join(repeats_folder, file), "r") as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) >= 2:
                    labels.append(parts[0])
                    repeats.append(parts[1])
    known_repeats = list(set(r for r in repeats if "X" not in r))
    known_repeats = list(set(r for r in known_repeats if "N" not in r))
    # -----------------------------
    # 2️⃣ Load train set
    # -----------------------------
    sequencesTrain, labelTrain = [], []
    for file in os.listdir(train_folder):
        if file.startswith("."):
            continue
        path = os.path.join(train_folder, file)
        records = SeqIO.parse(path, "fasta")
        for record in records:
            sequencesTrain.append(str(record.seq))
            labelTrain.append(file.replace(".fasta", ""))

    # -----------------------------
    # 3️⃣ Load test set
    # -----------------------------
    sequencesTest, labelTest = [], []
    for file in os.listdir(test_folder):
        if file.startswith("."):
            continue
        path = os.path.join(test_folder, file)
        records = SeqIO.parse(path, "fasta")
        for record in records:
            sequencesTest.append(str(record.seq))
            labelTest.append(file.replace(".fasta", ""))

    # -----------------------------
    # 4️⃣ Feature extraction
    # -----------------------------
    def extract_repeat_counts(sequence, repeat_list):
        return [sequence.count(r) for r in repeat_list]

    X_train = [extract_repeat_counts(seq, known_repeats) for seq in sequencesTrain]
    X_test = [extract_repeat_counts(seq, known_repeats) for seq in sequencesTest]

    # -----------------------------
    # 5️⃣ Encode labels
    # -----------------------------
    encoder = LabelEncoder()
    y_train_enc = encoder.fit_transform(labelTrain)
    y_test_enc = encoder.transform(labelTest)

    # -----------------------------
    # 6️⃣ Train KNN
    # -----------------------------
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric="minkowski")
    knn.fit(X_train, y_train_enc)

    # -----------------------------
    # 7️⃣ Predict
    # -----------------------------
    y_pred_enc = knn.predict(X_test)
    y_pred = encoder.inverse_transform(y_pred_enc)

    y_pred_train_enc = knn.predict(X_train)
    y_pred_train = encoder.inverse_transform(y_pred_train_enc)

    # -----------------------------
    # 8️⃣ Report
    # -----------------------------
    report = classification_report(labelTest, y_pred)
    print("Classification report:\n", report)
    print("Accuracy score:", accuracy_score(labelTest, y_pred))
    print("Accuracy score train:", accuracy_score(labelTrain, y_pred_train))

    with open(report_file, "w") as f:
        f.write(report)
        f.write("\nAccuracy score: " + str(accuracy_score(labelTest, y_pred)))
        f.write("\nAccuracy score train: " + str(accuracy_score(labelTrain, y_pred_train)))


# 🔹 Example usage:
# run_knn_with_features("../proba2NoDuplicates/indirect",
#                       "../trainSetNoDuplicates",
#                       "../testSetNoDuplicates",
#                       "classification_report_knn.txt", n_neighbors=5)


In [6]:
run_knn_with_features("../proba2NoDuplicates/indirect",
                       "../trainSetNoDuplicates",
                       "../testSetNoDuplicates",
                       "classification_report_knn_indirect_no_duplicates_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      0.99      0.99        94
   HCoV-HKU1       1.00      0.93      0.96        56
   HCoV-NL63       1.00      1.00      1.00       103
   HCoV-OC43       1.00      0.99      1.00       196
         IBV       1.00      1.00      1.00      2097
    MERS-CoV       1.00      0.99      0.99       138
    SARS-CoV       1.00      1.00      1.00         2
   SARS-CoV2       1.00      0.99      1.00       661
     bat-CoV       1.00      1.00      1.00        12
  bovine-CoV       0.98      1.00      0.99       245
  canine-CoV       0.84      0.97      0.90       143
 dolphin-CoV       1.00      1.00      1.00         2
  equine-CoV       1.00      0.86      0.92         7
  feline-CoV       0.97      0.96      0.97       389
  ferret-CoV       1.00      0.67      0.80         9
hedgehog-CoV       1.00      1.00      1.00         5
 porcine-CoV       1.00      1.00      1.00        99
  r

In [7]:
run_knn_with_features("../proba2NoDuplicates/direct",
                       "../trainSetNoDuplicates",
                       "../testSetNoDuplicates",
                       "classification_report_knn_direct_no_duplicates_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      0.99      0.99        94
   HCoV-HKU1       1.00      0.98      0.99        56
   HCoV-NL63       1.00      1.00      1.00       103
   HCoV-OC43       0.99      0.99      0.99       196
         IBV       1.00      1.00      1.00      2097
    MERS-CoV       1.00      0.98      0.99       138
    SARS-CoV       1.00      1.00      1.00         2
   SARS-CoV2       1.00      1.00      1.00       661
     bat-CoV       1.00      1.00      1.00        12
  bovine-CoV       0.99      1.00      0.99       245
  canine-CoV       0.93      0.95      0.94       143
 dolphin-CoV       1.00      1.00      1.00         2
  equine-CoV       1.00      0.86      0.92         7
  feline-CoV       0.96      0.99      0.97       389
  ferret-CoV       1.00      0.89      0.94         9
hedgehog-CoV       1.00      1.00      1.00         5
 porcine-CoV       1.00      1.00      1.00        99
  r

In [8]:
run_knn_with_features("../proba2/indirect",
                       "../trainSet",
                       "../testSet",
                       "classification_report_knn_indirect_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      1.00      1.00       153
   HCoV-HKU1       1.00      0.99      0.99       100
   HCoV-NL63       1.00      1.00      1.00       213
   HCoV-OC43       1.00      0.99      0.99       366
         IBV       1.00      1.00      1.00      3149
    MERS-CoV       0.98      0.99      0.99       395
    SARS-CoV       1.00      1.00      1.00         3
   SARS-CoV2       0.98      1.00      0.99      2133
     bat-CoV       1.00      1.00      1.00        15
  bovine-CoV       0.99      1.00      0.99       375
  canine-CoV       0.92      0.96      0.94       217
 dolphin-CoV       1.00      1.00      1.00         3
  equine-CoV       1.00      0.88      0.93         8
  feline-CoV       0.99      0.90      0.94       536
  ferret-CoV       1.00      0.77      0.87        13
hedgehog-CoV       1.00      1.00      1.00         6
 porcine-CoV       1.00      1.00      1.00       132
  r

In [9]:
run_knn_with_features("../proba2/direct",
                       "../trainSet",
                       "../testSet",
                       "classification_report_knn_direct_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      1.00      1.00       153
   HCoV-HKU1       1.00      1.00      1.00       100
   HCoV-NL63       1.00      1.00      1.00       213
   HCoV-OC43       1.00      0.99      1.00       366
         IBV       1.00      1.00      1.00      3149
    MERS-CoV       1.00      1.00      1.00       395
    SARS-CoV       1.00      1.00      1.00         3
   SARS-CoV2       1.00      1.00      1.00      2133
     bat-CoV       1.00      1.00      1.00        15
  bovine-CoV       0.99      1.00      1.00       375
  canine-CoV       0.89      0.96      0.92       217
 dolphin-CoV       1.00      1.00      1.00         3
  equine-CoV       1.00      0.88      0.93         8
  feline-CoV       0.98      0.98      0.98       536
  ferret-CoV       1.00      0.92      0.96        13
hedgehog-CoV       1.00      1.00      1.00         6
 porcine-CoV       1.00      0.99      1.00       132
  r

In [10]:
run_knn_with_features("../proba2NoDuplicatesNucl/DC",
                       "../trainSetNoDuplicatesNucl",
                       "../testSetNoDuplicatesNucl",
                       "classification_report_knn_DC_no_duplicates_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      0.99      1.00       120
   HCoV-HKU1       1.00      0.96      0.98        70
   HCoV-NL63       1.00      1.00      1.00       154
   HCoV-OC43       1.00      1.00      1.00       252
         IBV       1.00      0.99      1.00      2267
    MERS-CoV       1.00      0.97      0.98       198
    SARS-CoV       0.67      1.00      0.80         2
   SARS-CoV2       0.93      1.00      0.97       624
     bat-CoV       1.00      0.92      0.96        13
  bovine-CoV       1.00      1.00      1.00       276
  canine-CoV       0.99      0.90      0.94       162
 dolphin-CoV       1.00      1.00      1.00         2
  equine-CoV       1.00      0.86      0.92         7
  feline-CoV       0.96      0.97      0.96       444
  ferret-CoV       1.00      0.82      0.90        11
hedgehog-CoV       1.00      1.00      1.00         5
 porcine-CoV       1.00      1.00      1.00       108
  r

In [11]:
run_knn_with_features("../proba2NoDuplicatesNucl/DN",
                       "../trainSetNoDuplicatesNucl",
                       "../testSetNoDuplicatesNucl",
                       "classification_report_knn_DN_no_duplicates_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      0.99      1.00       120
   HCoV-HKU1       1.00      0.97      0.99        70
   HCoV-NL63       1.00      1.00      1.00       154
   HCoV-OC43       1.00      1.00      1.00       252
         IBV       1.00      0.99      0.99      2267
    MERS-CoV       1.00      0.97      0.98       198
    SARS-CoV       1.00      1.00      1.00         2
   SARS-CoV2       0.91      1.00      0.96       624
     bat-CoV       1.00      1.00      1.00        13
  bovine-CoV       0.99      1.00      1.00       276
  canine-CoV       0.99      0.88      0.93       162
 dolphin-CoV       1.00      1.00      1.00         2
  equine-CoV       1.00      0.86      0.92         7
  feline-CoV       0.98      0.97      0.97       444
  ferret-CoV       1.00      0.91      0.95        11
hedgehog-CoV       1.00      1.00      1.00         5
 porcine-CoV       1.00      1.00      1.00       108
  r

In [12]:
run_knn_with_features("../proba2NoDuplicatesNucl/IC",
                       "../trainSetNoDuplicatesNucl",
                       "../testSetNoDuplicatesNucl",
                       "classification_report_knn_IC_no_duplicates_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      0.99      1.00       120
   HCoV-HKU1       1.00      0.97      0.99        70
   HCoV-NL63       1.00      1.00      1.00       154
   HCoV-OC43       1.00      1.00      1.00       252
         IBV       1.00      0.99      0.99      2267
    MERS-CoV       1.00      0.98      0.99       198
    SARS-CoV       0.67      1.00      0.80         2
   SARS-CoV2       0.92      1.00      0.96       624
     bat-CoV       1.00      0.92      0.96        13
  bovine-CoV       0.99      1.00      1.00       276
  canine-CoV       0.97      0.89      0.93       162
 dolphin-CoV       1.00      1.00      1.00         2
  equine-CoV       1.00      0.86      0.92         7
  feline-CoV       0.98      0.97      0.97       444
  ferret-CoV       1.00      1.00      1.00        11
hedgehog-CoV       1.00      1.00      1.00         5
 porcine-CoV       1.00      1.00      1.00       108
  r

In [15]:
run_knn_with_features("../proba2NoDuplicatesNucl/IN",
                       "../trainSetNoDuplicatesNucl",
                       "../testSetNoDuplicatesNucl",
                       "classification_report_knn_IN_no_duplicates_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      0.99      1.00       120
   HCoV-HKU1       1.00      0.97      0.99        70
   HCoV-NL63       1.00      1.00      1.00       154
   HCoV-OC43       1.00      1.00      1.00       252
         IBV       1.00      0.99      1.00      2267
    MERS-CoV       1.00      0.97      0.98       198
    SARS-CoV       1.00      1.00      1.00         2
   SARS-CoV2       0.92      1.00      0.96       624
     bat-CoV       1.00      0.92      0.96        13
  bovine-CoV       0.99      1.00      0.99       276
  canine-CoV       0.98      0.86      0.92       162
 dolphin-CoV       1.00      1.00      1.00         2
  equine-CoV       1.00      0.86      0.92         7
  feline-CoV       0.97      0.98      0.97       444
  ferret-CoV       1.00      0.91      0.95        11
hedgehog-CoV       0.83      1.00      0.91         5
 porcine-CoV       1.00      1.00      1.00       108
  r

In [16]:
run_knn_with_features("../proba3/IC",
                       "../trainSetNucl",
                       "../testSetNucl",
                       "classification_report_knn_IC_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      0.97      0.98       153
   HCoV-HKU1       1.00      1.00      1.00       100
   HCoV-NL63       1.00      1.00      1.00       213
   HCoV-OC43       1.00      1.00      1.00       366
         IBV       1.00      0.99      1.00      3149
    MERS-CoV       1.00      0.99      0.99       395
    SARS-CoV       1.00      1.00      1.00         3
   SARS-CoV2       1.00      1.00      1.00      2133
     bat-CoV       1.00      0.80      0.89        15
  bovine-CoV       0.99      1.00      1.00       375
  canine-CoV       0.99      0.94      0.96       217
 dolphin-CoV       1.00      1.00      1.00         3
  equine-CoV       1.00      0.50      0.67         8
  feline-CoV       0.92      0.99      0.96       536
  ferret-CoV       1.00      0.92      0.96        13
hedgehog-CoV       1.00      1.00      1.00         6
 porcine-CoV       1.00      1.00      1.00       132
  r

In [17]:
run_knn_with_features("../proba3/IN",
                       "../trainSetNucl",
                       "../testSetNucl",
                       "classification_report_knn_IN_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      0.97      0.98       153
   HCoV-HKU1       1.00      1.00      1.00       100
   HCoV-NL63       1.00      1.00      1.00       213
   HCoV-OC43       0.99      1.00      1.00       366
         IBV       1.00      0.99      1.00      3149
    MERS-CoV       1.00      0.97      0.99       395
    SARS-CoV       1.00      1.00      1.00         3
   SARS-CoV2       1.00      1.00      1.00      2133
     bat-CoV       1.00      0.80      0.89        15
  bovine-CoV       0.99      1.00      1.00       375
  canine-CoV       0.99      0.93      0.96       217
 dolphin-CoV       1.00      1.00      1.00         3
  equine-CoV       1.00      0.50      0.67         8
  feline-CoV       0.89      0.99      0.94       536
  ferret-CoV       0.86      0.92      0.89        13
hedgehog-CoV       1.00      1.00      1.00         6
 porcine-CoV       1.00      0.99      1.00       132
  r

In [18]:
run_knn_with_features("../proba3/DC",
                       "../trainSetNucl",
                       "../testSetNucl",
                       "classification_report_knn_DC_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      0.97      0.98       153
   HCoV-HKU1       1.00      1.00      1.00       100
   HCoV-NL63       1.00      1.00      1.00       213
   HCoV-OC43       0.99      0.99      0.99       366
         IBV       1.00      0.99      1.00      3149
    MERS-CoV       1.00      0.98      0.99       395
    SARS-CoV       1.00      1.00      1.00         3
   SARS-CoV2       1.00      1.00      1.00      2133
     bat-CoV       1.00      0.87      0.93        15
  bovine-CoV       0.99      1.00      1.00       375
  canine-CoV       0.98      0.94      0.96       217
 dolphin-CoV       1.00      1.00      1.00         3
  equine-CoV       1.00      0.50      0.67         8
  feline-CoV       0.89      0.99      0.94       536
  ferret-CoV       1.00      0.77      0.87        13
hedgehog-CoV       1.00      1.00      1.00         6
 porcine-CoV       1.00      1.00      1.00       132
  r

In [19]:
run_knn_with_features("../proba3/DN",
                       "../trainSetNucl",
                       "../testSetNucl",
                       "classification_report_knn_DN_2.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      0.98      0.99       153
   HCoV-HKU1       1.00      1.00      1.00       100
   HCoV-NL63       1.00      1.00      1.00       213
   HCoV-OC43       0.99      1.00      1.00       366
         IBV       1.00      1.00      1.00      3149
    MERS-CoV       1.00      1.00      1.00       395
    SARS-CoV       1.00      1.00      1.00         3
   SARS-CoV2       1.00      1.00      1.00      2133
     bat-CoV       1.00      0.87      0.93        15
  bovine-CoV       0.99      1.00      1.00       375
  canine-CoV       0.92      0.98      0.95       217
 dolphin-CoV       1.00      1.00      1.00         3
  equine-CoV       1.00      0.50      0.67         8
  feline-CoV       0.99      0.99      0.99       536
  ferret-CoV       0.72      1.00      0.84        13
hedgehog-CoV       1.00      1.00      1.00         6
 porcine-CoV       1.00      0.99      1.00       132
  r

In [1]:
from collections import Counter
from Bio import SeqIO
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

def run_knn_with_features(repeats_folder, train_folder, test_folder, report_file, n_neighbors=5):
    # -----------------------------
    # 1️⃣ Load known repeats
    # -----------------------------
    labels, repeats = [], []
    for file in os.listdir(repeats_folder):
        if file.startswith("."):
            continue
        with open(os.path.join(repeats_folder, file), "r") as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) >= 2:
                    labels.append(parts[0])
                    repeats.append(parts[1])
    known_repeats = list(set(r for r in repeats if "X" not in r))
    known_repeats = list(set(r for r in known_repeats if "N" not in r))
    # -----------------------------
    # 2️⃣ Load train set
    # -----------------------------
    sequencesTrain, labelTrain = [], []
    for file in os.listdir(train_folder):
        if file.startswith("."):
            continue
        path = os.path.join(train_folder, file)
        records = SeqIO.parse(path, "fasta")
        for record in records:
            sequencesTrain.append(str(record.seq))
            labelTrain.append(file.replace(".fasta", ""))

    # -----------------------------
    # 3️⃣ Load test set
    # -----------------------------
    sequencesTest, labelTest = [], []
    for file in os.listdir(test_folder):
        if file.startswith("."):
            continue
        path = os.path.join(test_folder, file)
        records = SeqIO.parse(path, "fasta")
        for record in records:
            sequencesTest.append(str(record.seq))
            labelTest.append(file.replace(".fasta", ""))

    # -----------------------------
    # 4️⃣ Feature extraction
    # -----------------------------
    def extract_repeat_counts(sequence, repeat_list):
        return [sequence.count(r) for r in repeat_list]

    X_train = [extract_repeat_counts(seq, known_repeats) for seq in sequencesTrain]
    X_test = [extract_repeat_counts(seq, known_repeats) for seq in sequencesTest]

    # -----------------------------
    # 5️⃣ Encode labels
    # -----------------------------
    encoder = LabelEncoder()
    y_train_enc = encoder.fit_transform(labelTrain)
    y_test_enc = encoder.transform(labelTest)

    # -----------------------------
    # 6️⃣ Train KNN
    # -----------------------------
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric="minkowski")
    knn.fit(X_train, y_train_enc)

    # -----------------------------
    # 7️⃣ Predict
    # -----------------------------
    y_pred_enc = knn.predict(X_test)
    y_pred = encoder.inverse_transform(y_pred_enc)

    y_pred_train_enc = knn.predict(X_train)
    y_pred_train = encoder.inverse_transform(y_pred_train_enc)

    # -----------------------------
    # 8️⃣ Report
    # -----------------------------
    report = classification_report(labelTest, y_pred)
    print("Classification report:\n", report)
    print("Accuracy score:", accuracy_score(labelTest, y_pred))
    print("Accuracy score train:", accuracy_score(labelTrain, y_pred_train))

    with open(report_file, "w") as f:
        f.write(report)
        f.write("\nAccuracy score: " + str(accuracy_score(labelTest, y_pred)))
        f.write("\nAccuracy score train: " + str(accuracy_score(labelTrain, y_pred_train)))


# 🔹 Example usage:
# run_knn_with_features("../proba2NoDuplicates/indirect",
#                       "../trainSetNoDuplicates",
#                       "../testSetNoDuplicates",
#                       "classification_report_knn.txt", n_neighbors=5)


In [2]:
run_knn_with_features("../proba2/indirect",
                       "../trainSet",
                       "../testSet",
                       "proba2Ponovka_KNN_indirect.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      1.00      1.00       153
   HCoV-HKU1       1.00      0.99      0.99       100
   HCoV-NL63       1.00      1.00      1.00       213
   HCoV-OC43       1.00      0.99      0.99       366
         IBV       1.00      1.00      1.00      3149
    MERS-CoV       0.98      0.99      0.99       395
    SARS-CoV       1.00      1.00      1.00         3
   SARS-CoV2       0.98      1.00      0.99      2133
     bat-CoV       1.00      1.00      1.00        15
  bovine-CoV       0.99      1.00      0.99       375
  canine-CoV       0.92      0.96      0.94       217
 dolphin-CoV       1.00      1.00      1.00         3
  equine-CoV       1.00      0.88      0.93         8
  feline-CoV       0.99      0.90      0.94       536
  ferret-CoV       1.00      0.77      0.87        13
hedgehog-CoV       1.00      1.00      1.00         6
 porcine-CoV       1.00      1.00      1.00       132
  r

In [3]:
run_knn_with_features("../proba2/direct",
                       "../trainSet",
                       "../testSet",
                       "proba2Ponovka_KNN_direct.txt", n_neighbors=3)

Classification report:
               precision    recall  f1-score   support

   HCoV-229E       1.00      1.00      1.00       153
   HCoV-HKU1       1.00      1.00      1.00       100
   HCoV-NL63       1.00      1.00      1.00       213
   HCoV-OC43       1.00      0.99      1.00       366
         IBV       1.00      1.00      1.00      3149
    MERS-CoV       1.00      1.00      1.00       395
    SARS-CoV       1.00      1.00      1.00         3
   SARS-CoV2       1.00      1.00      1.00      2133
     bat-CoV       1.00      1.00      1.00        15
  bovine-CoV       0.99      1.00      1.00       375
  canine-CoV       0.89      0.96      0.92       217
 dolphin-CoV       1.00      1.00      1.00         3
  equine-CoV       1.00      0.88      0.93         8
  feline-CoV       0.98      0.98      0.98       536
  ferret-CoV       1.00      0.92      0.96        13
hedgehog-CoV       1.00      1.00      1.00         6
 porcine-CoV       1.00      0.99      1.00       132
  r